# 4. Data Prep: Build train / val / test from raw cohort data

This notebook does all the **heavy, one-time work**: unzip → load → merge → feature engineer → clean → split into **train / val / test**, then saves each split to a small parquet file. Run this notebook once (or whenever the raw data changes); after that, the modeling notebook (`5_modeling.ipynb`) just loads the finished parquet files and
never has to repeat this expensive merge again.

**Where outputs are saved:** `train.parquet` / `val.parquet` / `test.parquet` are written to `processed/` inside `DATA_DIR` on Drive, so they persist across Colab sessions.

**Split design**:
- **train**: all cohorts before `VAL_START` — used for time-series CV. The time-series CV is used as a robustness diagnostic, not for selection: it tells us whether a model is stable across many time points. Ideally the val winner also looks stable in CV; if the val winner swings wildly across CV folds, that's a flag to prefer a steadier model.
- **val**: cohorts from `VAL_START` up to `HOLDOUT_CUTOFF` — Each model is trained on the train set and scored on the fixed validation cohorts (Oct/Nov 2016). It's a clean split that sits in time after train, mimicking real deployment (train on history, predict the next period).
- **test**: cohorts from `HOLDOUT_CUTOFF` onward (Dec 2016 - Feb 2017) — save out the last 3 cohorts as a final test. Touched exactly once, at the very end, in the modeling notebook.

## 1. Imports & Config

In [1]:
import numpy as np
import pandas as pd
import os
import zipfile
import gc
from datetime import datetime

pd.set_option('display.max_columns', 100)


In [ ]:
#Run this if running in colab
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Churn Prediction/Data/"

In [2]:
#Run if on local machine
DATA_DIR = "csv_files/"

In [3]:

# write processed outputs back to Drive for persistence
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

SCRATCH_DIR = "/content/scratch"  # local disk scratch space for val/test spill-over, not Drive

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SCRATCH_DIR, exist_ok=True)

# Cohort cutoffs (monthly cohorts run Feb 2015 - Feb 2017)
VAL_START = datetime(2016, 10, 1)        # val = Oct/Nov 2016 (2 cohorts)
HOLDOUT_CUTOFF = datetime(2016, 12, 1)   # test = Dec 2016 - Feb 2017 (3 cohorts)

# features come from notenoke 3_EDA
FEATURES = [
    'avg_plan_price',
    'avg_payment_per_day',
    'days_since_first_trans',
    'days_since_last_use',
    'ratio_auto_renew',
    'total_secs_velocity',
    'num_unq_velocity',
    'last_is_auto_renew',
    'last_is_cancel',
]
MISSING_FLAG_COLS = ['days_since_last_use_missing', 'total_secs_velocity_missing', 'num_unq_velocity_missing']
TARGET = 'is_churn'
KEEP_COLS = ['msno', 'cohort_cutoff_date'] + FEATURES + MISSING_FLAG_COLS + [TARGET]


## 2. Auto-Unzip Raw Data

In [4]:
def ensure_extracted(zip_filename, expected_filename, data_dir=DATA_DIR):
    expected_path = os.path.join(data_dir, expected_filename)
    if os.path.exists(expected_path):
        print(f"Found {expected_filename}, skipping extraction.")
        return expected_path

    zip_path = os.path.join(data_dir, zip_filename)
    if not os.path.exists(zip_path):
        raise FileNotFoundError(
            f"Neither {expected_filename} nor {zip_filename} found in {data_dir}. "
            f"Contents of {data_dir}: {os.listdir(data_dir)}"
        )

    print(f"Extracting {zip_filename}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(data_dir)
    print(f"Done. {data_dir} now contains: {os.listdir(data_dir)}")
    return expected_path


trans_csv_path = ensure_extracted("mult_cohort_transaction_data.zip", "mult_cohort_transaction_data.csv")
usage_parquet_path = ensure_extracted("mult_cohort_usage_data.zip", "mult_cohort_usage_data.parquet")


Found mult_cohort_transaction_data.csv, skipping extraction.
Found mult_cohort_usage_data.parquet, skipping extraction.


## 3. Load Raw Data

In [5]:
trans_data = pd.read_csv(
    trans_csv_path,
    parse_dates=["last_transaction_date", "cohort_cutoff_date"]
)

usage_data = pd.read_parquet(usage_parquet_path)
usage_data["cohort_cutoff_date"] = pd.to_datetime(usage_data["cohort_cutoff_date"])

print("Transactions shape:", trans_data.shape)
print("Usage shape:", usage_data.shape)

Transactions shape: (16857930, 19)
Usage shape: (16857930, 9)


## 4. Memory Optimization

Shared `msno` categories across all two frames (avoids the merge-time string-decode blowup), plus downcasting numeric dtypes, plus dropping `bd`/`registration_init_time` early.


In [6]:
def downcast_numeric(frame):
    for col in frame.select_dtypes(include=['int64']).columns:
        frame[col] = pd.to_numeric(frame[col], downcast='integer')
    for col in frame.select_dtypes(include=['float64']).columns:
        frame[col] = pd.to_numeric(frame[col], downcast='float')
    return frame

msno_categories = pd.unique(np.concatenate([
    trans_data['msno'].astype(str).unique(),
    usage_data['msno'].astype(str).unique(),
]))
msno_dtype = pd.CategoricalDtype(categories=msno_categories)

trans_data['msno'] = trans_data['msno'].astype(str).astype(msno_dtype)
usage_data['msno'] = usage_data['msno'].astype(str).astype(msno_dtype)

trans_data = downcast_numeric(trans_data)
usage_data = downcast_numeric(usage_data)

gc.collect()
print("Shared msno categories:", len(msno_categories))

Shared msno categories: 1917985


## 5. Split Into Three Time Ranges (before merging)

We split `trans_data`/`usage_data` by `cohort_cutoff_date` into train/val/test **before** merging them together. Only the **train** slice stays in memory right away; val and test are spilled to scratch disk and reloaded later, one at a time, when we actually process them. This keeps peak memory bounded to roughly one split's worth of data.

In [7]:
def three_way_split(frame, date_col='cohort_cutoff_date'):
    train = frame.loc[frame[date_col] < VAL_START]
    val = frame.loc[(frame[date_col] >= VAL_START) & (frame[date_col] < HOLDOUT_CUTOFF)]
    test = frame.loc[frame[date_col] >= HOLDOUT_CUTOFF]
    return train, val, test


trans_train, trans_val, trans_test = three_way_split(trans_data)
del trans_data
gc.collect()

usage_train, usage_val, usage_test = three_way_split(usage_data)
del usage_data
gc.collect()

print("trans  -> train:", trans_train.shape, "val:", trans_val.shape, "test:", trans_test.shape)
print("usage  -> train:", usage_train.shape, "val:", usage_val.shape, "test:", usage_test.shape)

# Save as parquet (not csv): smaller on disk, faster to read, and preserves column dtypes, ideal for these write-once, read-many intermediate files.
# Spill val/test slices to scratch disk and free them from memory
# we only work on one split at a time, starting with train.
trans_val.to_parquet(os.path.join(SCRATCH_DIR, "trans_val.parquet"))
usage_val.to_parquet(os.path.join(SCRATCH_DIR, "usage_val.parquet"))
trans_test.to_parquet(os.path.join(SCRATCH_DIR, "trans_test.parquet"))
usage_test.to_parquet(os.path.join(SCRATCH_DIR, "usage_test.parquet"))

del trans_val, usage_val, trans_test, usage_test
gc.collect()
print("val/test raw slices written to scratch disk and freed from memory.")


trans  -> train: (12543036, 19) val: (1708636, 19) test: (2606258, 19)
usage  -> train: (12543036, 9) val: (1708636, 9) test: (2606258, 9)
val/test raw slices written to scratch disk and freed from memory.


## 6. Feature Engineering + Missing-Value Handling (shared function)

Same logic applied to all three splits, so there's no risk of treating them differently.
The imputation sentinel for `days_since_last_use` is computed from **train only** and then
reused for val/test, so we never leak information about val/test's usage distribution into
the imputed value.


In [8]:
def build_features(trans_subset, usage_subset):
    frame = pd.merge(trans_subset, usage_subset, on=['msno', 'cohort_cutoff_date'], how='left')

    frame['ratio_auto_renew'] = frame['total_auto_renew'] / frame['num_transactions']

    for col in ['days_since_last_use', 'total_secs_velocity', 'num_unq_velocity']:
        frame[f'{col}_missing'] = frame[col].isna().astype(int)

    return frame


def impute_features(frame, max_days_since_last_use):
    frame['days_since_last_use'] = frame['days_since_last_use'].fillna(max_days_since_last_use + 1)
    frame['total_secs_velocity'] = frame['total_secs_velocity'].fillna(0)
    frame['num_unq_velocity'] = frame['num_unq_velocity'].fillna(0)
    return frame

### 6a. Build & save `train`

In [9]:
df_train = build_features(trans_train, usage_train)
del trans_train, usage_train
gc.collect()

max_days_since_last_use = df_train['days_since_last_use'].max()  # computed from TRAIN only
df_train = impute_features(df_train, max_days_since_last_use)

print("Train shape:", df_train.shape, "| churn rate:", df_train[TARGET].mean().round(4))
print("Train cohorts:", sorted(df_train['cohort_cutoff_date'].unique()))

df_train[KEEP_COLS].to_parquet(os.path.join(PROCESSED_DIR, "train.parquet"))
del df_train
gc.collect()
print("Saved train.parquet")

Train shape: (12543036, 30) | churn rate: 0.0796
Train cohorts: [Timestamp('2015-02-01 00:00:00'), Timestamp('2015-03-01 00:00:00'), Timestamp('2015-04-01 00:00:00'), Timestamp('2015-05-01 00:00:00'), Timestamp('2015-06-01 00:00:00'), Timestamp('2015-07-01 00:00:00'), Timestamp('2015-08-01 00:00:00'), Timestamp('2015-09-01 00:00:00'), Timestamp('2015-10-01 00:00:00'), Timestamp('2015-11-01 00:00:00'), Timestamp('2015-12-01 00:00:00'), Timestamp('2016-01-01 00:00:00'), Timestamp('2016-02-01 00:00:00'), Timestamp('2016-03-01 00:00:00'), Timestamp('2016-04-01 00:00:00'), Timestamp('2016-05-01 00:00:00'), Timestamp('2016-06-01 00:00:00'), Timestamp('2016-07-01 00:00:00'), Timestamp('2016-08-01 00:00:00'), Timestamp('2016-09-01 00:00:00')]
Saved train.parquet


### 6b. Build & save `val`

In [10]:
trans_val = pd.read_parquet(os.path.join(SCRATCH_DIR, "trans_val.parquet"))
usage_val = pd.read_parquet(os.path.join(SCRATCH_DIR, "usage_val.parquet"))

df_val = build_features(trans_val, usage_val)
del trans_val, usage_val
gc.collect()

df_val = impute_features(df_val, max_days_since_last_use)  # reuse TRAIN's sentinel

print("Val shape:", df_val.shape, "| churn rate:", df_val[TARGET].mean().round(4))
print("Val cohorts:", sorted(df_val['cohort_cutoff_date'].unique()))

df_val[KEEP_COLS].to_parquet(os.path.join(PROCESSED_DIR, "val.parquet"))
del df_val
gc.collect()
print("Saved val.parquet")

Val shape: (1708636, 30) | churn rate: 0.0652
Val cohorts: [Timestamp('2016-10-01 00:00:00'), Timestamp('2016-11-01 00:00:00')]
Saved val.parquet


### 6c. Build & save `test` (final holdout, not touched again until final evaluation)

In [11]:
trans_test = pd.read_parquet(os.path.join(SCRATCH_DIR, "trans_test.parquet"))
usage_test = pd.read_parquet(os.path.join(SCRATCH_DIR, "usage_test.parquet"))

df_test = build_features(trans_test, usage_test)
del trans_test, usage_test
gc.collect()

df_test = impute_features(df_test, max_days_since_last_use)  # reuse TRAIN's sentinel

print("Test shape:", df_test.shape, "| churn rate:", df_test[TARGET].mean().round(4))
print("Test cohorts:", sorted(df_test['cohort_cutoff_date'].unique()))

df_test[KEEP_COLS].to_parquet(os.path.join(PROCESSED_DIR, "test.parquet"))
del df_test
gc.collect()
print("Saved test.parquet")

Test shape: (2606258, 30) | churn rate: 0.0551
Test cohorts: [Timestamp('2016-12-01 00:00:00'), Timestamp('2017-01-01 00:00:00'), Timestamp('2017-02-01 00:00:00')]
Saved test.parquet


## 7. Done

The `processed/` folder now contains `train.parquet`, `val.parquet`, and `test.parquet`. Each already merged, feature-engineered, and cleaned, with only the columns the model actually needs (`msno`, `cohort_cutoff_date`, the 9 features, missing-value flags, and `is_churn`).

Open `5_modeling.ipynb` next, it loads these three files directly and never has to repeat the merge/cleaning steps in this notebook. Only re-run this notebook if the raw data changes or you want to add/change features.
